# Gaming Toxicity Detection

**Authors:** Beibarys Nyussupov, Ruby Ngo, Paola Calle, Jett Forward

In [1]:
# libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path 
import re 
import sys
sys.path.insert(0, str(Path("../..").resolve()))

# custom functions - tokenizer, step evaluator
from src.tokenizer import tokenize
from src.prep_evaluation_multi import eval_step

# reproducibility
seed = 7524
np.random.seed(seed)

# project root (notebooks/gaming/ - notebooks/ - project root)
PROJECT_ROOT = Path().resolve().parent.parent

In [2]:
# data directories
DATA_DIR_WOT  = PROJECT_ROOT / "data/raw_data/wot/"
DATA_DIR_DOTA = PROJECT_ROOT / "data/raw_data/dota/"

## World of Tanks

**World of Tanks**
| Class | Terminology |
|---|---|
| 0 | Non-Toxic |
| 1 | Insults and Flaming |
| 2 | Other Offensive Texts |
| 3 | Hate and Harassment |
| 4 | Threats |
| 5 | Extremism |

#### Inspect each split

In [3]:
# import data 
train = pd.read_csv(DATA_DIR_WOT / "train.csv")
val = pd.read_csv(DATA_DIR_WOT / "val.csv")

# test dataset 
test_text = pd.read_csv(DATA_DIR_WOT /  "test_index_text.csv")
test_label = pd.read_csv(DATA_DIR_WOT / "test_index_label.csv")

# check the data 
for name, df in [("train", train), ("val", val), ("test_text", test_text), ("test_label", test_label)]:
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"First few rows of the dataset:\n{df.head(3)}")
    print()

--- train ---
Shape: (42959, 3)
Columns: ['index', 'message', 'label']
First few rows of the dataset:
   index                        message  label
0  30702                        no rush    0.0
1  18607  whatever ... watch the replay    0.0
2  32901                        useless    1.0

--- val ---
Shape: (5367, 3)
Columns: ['index', 'message', 'label']
First few rows of the dataset:
   index       message  label
0  31697  e50 is there    0.0
1  52311   this scouts    0.0
2   2775       i guess    0.0

--- test_text ---
Shape: (5375, 2)
Columns: ['index', 'message']
First few rows of the dataset:
   index         message
0  17775  aim you monkey
1  28228   Pz kill artys
2  19888             ggs

--- test_label ---
Shape: (5375, 2)
Columns: ['index', 'label']
First few rows of the dataset:
   index  label
0  17775    2.0
1  28228    0.0
2  19888    0.0



The test set is stored in two separate files - text and labels indexed by the same `index` column. Train and val are already complete. We need to join the test files on `index` before merging everything.

#### Reconstruct test split and merge all

In [4]:
# merge test text and labels
test = test_text.merge(test_label, on="index", how="inner")

# concatenate all data
wot = pd.concat([train, val, test], ignore_index=True)

print(f"Test join shape: {test.shape}\n")
print(f"Merged shape: {wot.shape}\n")
print(f"First few rows of the data:\n{wot.head(3)}\n")
print(f"Information about the data: {wot.info()}")

Test join shape: (5375, 3)

Merged shape: (53701, 3)

First few rows of the data:
   index                        message  label
0  30702                        no rush    0.0
1  18607  whatever ... watch the replay    0.0
2  32901                        useless    1.0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53701 entries, 0 to 53700
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   index    53701 non-null  int64  
 1   message  53701 non-null  object 
 2   label    53701 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.2+ MB
Information about the data: None


The test join uses `inner` - any index present in only one file would be silently dropped. If test shape after join is smaller than either file's length, there is an alignment issue in the original data. 

In [5]:
# to test improvement or degradation after each preprocessing step, 
# we export the data as raw parquet and compare score of predictions of raw vs processed data. 
wot.to_parquet(PROJECT_ROOT / "data/processed_data/wot/wot.parquet", index=False)

# baseline — before any cleaning
preprocessing_impact = eval_step("baseline_raw", datasets = ("WoT",))

        step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
baseline_raw     WoT 53701    0.4855        0.861        0.5599           0.4696             0.0           0.0              0.0


#### Non-english language 

Since we are going to model classifier on english data, it is important for us to detect non-english cases as many as we can. 

In [6]:
# detecting Non-lating-script messages 

# covers all major non-Latin characters
NON_LATIN_SCRIPT = re.compile(
    r"[\u0400-\u04FF"   # Cyrillic
    r"\u4E00-\u9FFF"    # CJK unified ideographs
    r"\u3400-\u4DBF"    # CJK extension A
    r"\uF900-\uFAFF"    # CJK compatibility ideographs
    r"\u0600-\u06FF"    # Arabic
    r"\u0590-\u05FF"    # Hebrew
    r"\u3040-\u30FF"    # Japanese (Hiragana + Katakana)
    r"\uAC00-\uD7AF"    # Korean (Hangul syllables)
    r"\u1100-\u11FF"    # Korean (Hangul Jamo)
    r"\u0E00-\u0E7F"    # Thai
    r"\u0900-\u097F"    # Devanagari (Hindi)
    r"\u0980-\u09FF"    # Bengali
    r"\u0370-\u03FF"    # Greek
    r"\u10A0-\u10FF"    # Georgian
    r"\u0530-\u058F"    # Armenian
    r"\u1000-\u109F"    # Myanmar
    r"\u1780-\u17FF]"   # Khmer
)

# drop any message containing even 1 non-Latin script char
print(f"Non-english documents:\n{wot[wot['message'].str.contains(NON_LATIN_SCRIPT, regex=True, na=False)].reset_index(drop=True)}\n")

# changes 
before = len(wot)
wot_en = wot[~wot["message"].str.contains(NON_LATIN_SCRIPT, regex=True, na=False)].reset_index(drop=True)
print(f"Dropped {before - len(wot_en)} ({(before - len(wot_en))/before:.1%})")
print("Label distribution before:")
print(wot["label"].value_counts(normalize=True).sort_index(), "\n")

print("\nLabel distribution after:")
print(wot_en["label"].value_counts(normalize=True).sort_index(), "\n")
print(f"Rows after the drop: {wot_en.shape[0]}")
# save new step again
wot_en.to_parquet(PROJECT_ROOT / "data/processed_data/wot/wot.parquet", index=False)


Non-english documents:
      index                message  label
0     13087          ще злийте )))    1.0
1     31917       перший бій, сток    0.0
2     40395                ФОКУСЯТ    0.0
3     54412  елка спасибо тебе ...    0.0
4      5457                 уроди!    0.0
...     ...                    ...    ...
3845   5453                 я кину    0.0
3846   5456                светите    0.0
3847  13081       в чём притензия?    0.0
3848   7331      вот и подкруточки    0.0
3849  50128  тут танки стоят уебак    0.0

[3850 rows x 3 columns]

Dropped 3850 (7.2%)
Label distribution before:
label
0.0    0.809985
1.0    0.137930
2.0    0.043630
3.0    0.006499
4.0    0.001397
5.0    0.000559
Name: proportion, dtype: float64 


Label distribution after:
label
0.0    0.812782
1.0    0.136126
2.0    0.043770
3.0    0.005376
4.0    0.001384
5.0    0.000562
Name: proportion, dtype: float64 

Rows after the drop: 49851


In [7]:
# check non-latin language drop
preprocessing_impact = eval_step("after_nonlatin_drop", preprocessing_impact, datasets="WoT")
preprocessing_impact

               step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
after_nonlatin_drop     WoT 49851    0.4976       0.8692        0.5921           0.4799          0.0121        0.0322           0.0103


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4855,0.8610,0.5599,0.4696,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4976,0.8692,0.5921,0.4799,0.0121,0.0322,0.0103


In [8]:
import re
import pandas as pd
from lingua import Language, LanguageDetectorBuilder

# detecting Latin-script but likely non-English messages

# use all Latin-script languages supported by Lingua
# English must stay inside the detector, otherwise English text gets forced into another language
LATIN_LANGUAGES = list(Language.all_with_latin_script())

# detector for Latin-script language detection
# minimum_relative_distance: if top languages are too close, Lingua returns None instead of guessing
detector = (
    LanguageDetectorBuilder
    .from_languages(*LATIN_LANGUAGES)
    .with_minimum_relative_distance(0.25)
    .build()
)

# safety tokens: if these appear, keep the message because it may be gaming/chat English or mixed slang
SAFETY_TOKENS = {
    # gaming / chat
    "gg", "ez", "wp", "gl", "hf", "glhf", "afk", "brb", "lol", "lmao", "lmfao",
    "omg", "wtf", "stfu", "noob", "n00b", "bot", "team", "report", "ban",
    "kick", "mute", "trash", "toxic", "troll", "feed", "feeder", "int", "throw",
    "rank", "elo", "kill", "kys", "die", "fps", "lag", "ping", "server",
    "wot", "dota", "fortnite", "fn",

    # English / toxicity cues
    "you", "your", "you're", "u", "ur", "are", "is", "this", "that", "the",
    "and", "not", "dont", "don't", "fuck", "fucking", "fuk", "fck", "shit",
    "bitch", "ass", "asshole", "idiot", "stupid", "dumb", "moron", "loser",
    "suck", "sucks"
}

TOKEN_RE = re.compile(r"[a-z0-9']+", flags=re.IGNORECASE)


def get_tokens(text):
    return set(TOKEN_RE.findall(str(text).lower()))


def has_safety_token(text):
    return bool(get_tokens(text) & SAFETY_TOKENS)


# function to classify what to do with each Latin-script message
def latin_language_filter_status(text):
    text_str = str(text).strip()
    tokens = get_tokens(text_str)

    # too short - keep
    # language detection is unreliable on short gaming/chat messages
    if len(text_str) < 50 or len(tokens) < 6:
        return "keep_short"

    # possible gaming/chat/English false positive - keep
    if has_safety_token(text_str):
        return "keep_possible_gaming_or_mixed"

    lang = detector.detect_language_of(text_str)

    # ambiguous - keep
    if lang is None:
        return "keep_ambiguous"

    # English - keep
    if lang == Language.ENGLISH:
        return "keep_english"

    # confident Latin-script non-English - drop
    return f"drop_latin_nonenglish_{lang.name.lower()}"


# apply detection
wot_en_lang = wot_en.copy()
wot_en_lang["lang_filter_status"] = wot_en_lang["message"].apply(latin_language_filter_status)

# inspect groups
print("Language filter status counts:")
print(wot_en_lang["lang_filter_status"].value_counts().head(30))


# check ambiguous cases
# these are messages where Lingua did not confidently choose a language
ambiguous_mask = wot_en_lang["lang_filter_status"].eq("keep_ambiguous")

print(f"\nAmbiguous Latin-script language documents kept: {ambiguous_mask.sum()}")

with pd.option_context("display.max_rows", 300, "display.max_colwidth", 180):
    print(
        "\nAmbiguous documents kept for now:\n",
        wot_en_lang.loc[
            ambiguous_mask,
            ["message", "label", "lang_filter_status"]
        ]
    )

print("\nLabel distribution among ambiguous rows:")
print(
    wot_en_lang.loc[ambiguous_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)


# check possible gaming / mixed-language false positives
# these are messages that Lingua might classify as non-English, but contain gaming/chat/English cues
mixed_mask = wot_en_lang["lang_filter_status"].eq("keep_possible_gaming_or_mixed")

print(f"\nPossible gaming or mixed-language documents kept: {mixed_mask.sum()}")

with pd.option_context("display.max_rows", 300, "display.max_colwidth", 180):
    print(
        "\nPossible gaming/mixed documents kept for now:\n",
        wot_en_lang.loc[
            mixed_mask,
            ["message", "label", "lang_filter_status"]
        ]
    )

print("\nLabel distribution among possible gaming/mixed rows:")
print(
    wot_en_lang.loc[mixed_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)


# drop only confident Latin-script non-English rows
drop_mask = wot_en_lang["lang_filter_status"].str.startswith("drop_latin_nonenglish")

with pd.option_context("display.max_rows", 300, "display.max_colwidth", 180):
    print(
        "\nConfident Latin-script non-English documents to drop:\n",
        wot_en_lang.loc[
            drop_mask,
            ["message", "label", "lang_filter_status"]
        ]
    )

print(f"\nHow many confident Latin-script non-English documents: {drop_mask.sum()}")


# label distribution sanity check
dropped_latin_nonenglish = wot_en_lang.loc[drop_mask]

print("\nLabel distribution before Latin non-English drop:")
print(wot_en["label"].value_counts(normalize=True).sort_index())

print("\nLabel distribution after Latin non-English drop:")
print(
    wot_en_lang.loc[~drop_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)

print("\nLabel distribution among dropped rows:")
print(
    dropped_latin_nonenglish["label"]
    .value_counts(normalize=True)
    .sort_index()
)


# create dataset after Latin non-English drop
# keep ambiguous and mixed/gaming rows
wot_lang_clean = (
    wot_en_lang
    .loc[~drop_mask]
    .drop(columns=["lang_filter_status"])
    .reset_index(drop=True)
)

print(f"\nRows before Latin non-English drop: {len(wot_en)}")
print(f"Dropped {drop_mask.sum()} ({drop_mask.sum() / len(wot_en):.1%})")
print(f"Rows after Latin non-English drop: {len(wot_lang_clean)}")

Language filter status counts:
lang_filter_status
keep_short                         48686
keep_possible_gaming_or_mixed        981
keep_english                          63
keep_ambiguous                        43
drop_latin_nonenglish_polish          29
drop_latin_nonenglish_german          26
drop_latin_nonenglish_hungarian        8
drop_latin_nonenglish_french           6
drop_latin_nonenglish_turkish          3
drop_latin_nonenglish_czech            2
drop_latin_nonenglish_spanish          1
drop_latin_nonenglish_bosnian          1
drop_latin_nonenglish_romanian         1
drop_latin_nonenglish_finnish          1
Name: count, dtype: int64

Ambiguous Latin-script language documents kept: 43

Ambiguous documents kept for now:
                                                                                        message  \
456                             I've gotten all city maps today so I was like &quot;FINE&quot;   
532               BattleGreen (Skorpion G), T'es tellement gros qu

In [9]:
# save to parquet
wot_lang_clean.to_parquet(PROJECT_ROOT / "data/processed_data/wot/wot.parquet", index=False)

# evaluate impact 
preprocessing_impact = eval_step("after_noneng_drop", preprocessing_impact, datasets = ("WoT",))
preprocessing_impact

             step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
after_noneng_drop     WoT 49773    0.4922       0.8683        0.5921           0.4788         -0.0054           0.0          -0.0011


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4855,0.8610,0.5599,0.4696,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4976,0.8692,0.5921,0.4799,0.0121,0.0322,0.0103
2,after_noneng_drop,WoT,49773,0.4922,0.8683,0.5921,0.4788,-0.0054,0.0000,-0.0011


Decreases in metrics are within noise. 

#### Duplicates

In [10]:
# look at dataset with duplicates 
wot_dup = wot_lang_clean[wot_lang_clean.duplicated(subset="message", keep=False)].sort_values("message")
# look at some examples
wot_dup.head(10)

,index,message,label
14235,31613,!,0.0
39883,6872,!,0.0
43647,40582,!,0.0
45639,1278,!!!!,0.0
27680,39948,!!!!,0.0
33470,32907,!!!!,0.0
9468,16596,#,0.0
11562,8848,#,0.0
2637,35311,#ERROR!,0.0
10519,23725,#ERROR!,0.0


In [11]:
# top 100 duplicated messages
wot_dup["message"].value_counts(ascending=False).head(100)

message
gg                   3188
lol                   458
nice                  396
gj                    315
wtf                   201
                     ... 
fu                     26
reported               26
??                     26
ggg                    26
Im Spotted at G2!      26
Name: count, Length: 100, dtype: int64

Most of these duplicates are useful and important data to train or models for detecting toxicity. We need to check for same messages but different labels.

In [12]:
# Look for duplicates with same messages but different labels
conflicts = wot_lang_clean.groupby("message")["label"].nunique()
conflicts = conflicts[conflicts > 1]

conflict_rows = wot_lang_clean[wot_lang_clean["message"].isin(conflicts.index)].sort_values("message")

# Check how large is the proportion of conflicting duplicates
conflict_pct = len(conflict_rows) / len(wot_lang_clean) * 100
print(f"Proportion of messages with conflicting labels: {conflict_pct:.2f}%")
print(f"Number of messages with conflicting labels: {len(conflict_rows)}")

Proportion of messages with conflicting labels: 14.66%
Number of messages with conflicting labels: 7298


In [13]:
# check conflicting rows 
conflict_rows 

,index,message,label
44262,24318,#ERROR!,0.0
10272,19840,#ERROR!,0.0
32084,13407,#ERROR!,0.0
36961,14302,#ERROR!,0.0
42479,14881,#ERROR!,0.0
...,...,...,...
21216,44232,you lost,2.0
21116,13279,you lost,1.0
45823,17220,you lost,0.0
5423,1393,you wanna loose that right?,1.0


These messages can not be used for analysis, since it is a problem of `annotation`. Using them will be equal to guessing. Duplicated rows account for 14% of the data, which is a huge loss but we can not use that.

In [14]:
# drop conflicting messages
wot_dup = wot_lang_clean[~wot_lang_clean["message"].isin(conflicts.index)].reset_index(drop=True)
print(wot_dup.shape)

(42475, 3)


In [15]:
# save again and evaluate 
wot_dup.to_parquet(PROJECT_ROOT / "data/processed_data/wot/wot.parquet", index=False)
preprocessing_impact = eval_step("after_duplicates_removal", preprocessing_impact, datasets = ("WoT",))


                    step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
after_duplicates_removal     WoT 42475    0.4919       0.8658        0.5711           0.5091         -0.0003        -0.021           0.0303


In [16]:
preprocessing_impact

,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4855,0.8610,0.5599,0.4696,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4976,0.8692,0.5921,0.4799,0.0121,0.0322,0.0103
2,after_noneng_drop,WoT,49773,0.4922,0.8683,0.5921,0.4788,-0.0054,0.0000,-0.0011
3,after_duplicates_removal,WoT,42475,0.4919,0.8658,0.5711,0.5091,-0.0003,-0.0210,0.0303


This did not work, our score started decreasing. Instead of dropping duplicated conflicting labels, we assign a majority label to each. 

In [17]:
# majority label for similar words 
majority_label = wot_lang_clean.groupby("message")["label"].agg(lambda x: x.value_counts().index[0])

# drop duplicates and map majority label 
wot_deduped = wot_lang_clean.copy()
wot_deduped["label"] = wot_deduped["message"].map(majority_label)

# check same ex duplicates 
wot_deduped[wot_deduped["message"].isin(conflicts.index)][["message", "label"]].head(20)

,message,label
4,lol,0.0
11,#ERROR!,0.0
18,bot,1.0
23,retard,1.0
46,Im Spotted at H6!,0.0
47,ty,0.0
53,ns,0.0
62,wtf,2.0
64,#ERROR!,0.0
85,idiots,1.0


In [18]:
# save again and evaluate 
wot_deduped.to_parquet(PROJECT_ROOT / "data/processed_data/wot/wot.parquet", index=False)
preprocessing_impact = eval_step("after_duplicates_majority_map", preprocessing_impact, datasets = ("WoT",))
preprocessing_impact

                         step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
after_duplicates_majority_map     WoT 49773    0.5291       0.8778        0.6233           0.5106          0.0372        0.0522           0.0015


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4855,0.8610,0.5599,0.4696,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4976,0.8692,0.5921,0.4799,0.0121,0.0322,0.0103
2,after_noneng_drop,WoT,49773,0.4922,0.8683,0.5921,0.4788,-0.0054,0.0000,-0.0011
3,after_duplicates_removal,WoT,42475,0.4919,0.8658,0.5711,0.5091,-0.0003,-0.0210,0.0303
4,after_duplicates_majority_map,WoT,49773,0.5291,0.8778,0.6233,0.5106,0.0372,0.0522,0.0015


That is the best we can do and it actually improves the score. 

#### Data Artifacts 


In [19]:
# #ERROR! entries
error_count = wot_deduped["message"].str.contains(r"#ERROR!", regex=False, na=False).sum()
print(f"#ERROR! messages: {error_count}")

# HTML entities
html_mask = wot_deduped["message"].str.contains(r"&\w+;", regex=True, na=False)
print(f"Messages with HTML entities: {html_mask.sum()}")
wot_deduped[html_mask][["message", "label"]].head(10)

#ERROR! messages: 171
Messages with HTML entities: 148


,message,label
24,&lt;#,0.0
128,&gt;bz,0.0
293,&lt;3,0.0
385,&lt;3,0.0
456,I've gotten all city maps today so I was like ...,0.0
553,"czech_mad_man[BD_CR] (Centurion 7/1), you‘re B...",0.0
1244,tog &amp; roll !! ladies,0.0
1625,&quot;_),0.0
2724,&lt;3,0.0
2781,&lt;3,0.0


In [20]:
import re
import html as html_lib
import pandas as pd

# artifact cleaning
# goal: remove generic technical artifacts, not game-specific phrases
# use wot_deduped as input from the previous preprocessing step

wot_clean = wot_deduped.copy()

# convert messages to string and fill missing values
wot_clean["message"] = wot_clean["message"].fillna("").astype(str)

# decode HTML entities inline
# example: &quot; -> ", &amp; -> &
wot_clean["message"] = wot_clean["message"].apply(html_lib.unescape)

# generic artifact patterns
# no game-specific hardcoding here
ERROR_RE = re.compile(r"#ERROR!", flags=re.IGNORECASE)
URL_RE = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
HTML_TAG_RE = re.compile(r"<[^>]+>", flags=re.IGNORECASE)

# after html.unescape, most entities should disappear
# this catches remaining malformed/unresolved entities
HTML_ENTITY_RE = re.compile(r"&[a-zA-Z]+;|&#\d+;|&#x[0-9a-fA-F]+;")


# classify artifact reason for each row
def artifact_status(text):
    text = str(text)

    if ERROR_RE.search(text):
        return "drop_error"

    if URL_RE.search(text):
        return "drop_url"

    if HTML_TAG_RE.search(text):
        return "drop_html_tag"

    if HTML_ENTITY_RE.search(text):
        return "drop_unresolved_html_entity"

    return "keep"


# apply artifact detection
wot_clean["artifact_status"] = wot_clean["message"].apply(artifact_status)

# inspect artifact counts
print("Artifact status counts:")
print(wot_clean["artifact_status"].value_counts())

# rows to drop
artifact_mask = wot_clean["artifact_status"].ne("keep")

# inspect dropped artifacts
with pd.option_context("display.max_rows", 300, "display.max_colwidth", 180):
    print(
        "\nArtifact rows to drop:\n",
        wot_clean.loc[
            artifact_mask,
            ["message", "label", "artifact_status"]
        ]
    )

# label distribution sanity check
print("\nLabel distribution before artifact drop:")
print(wot_deduped["label"].value_counts(normalize=True).sort_index())

print("\nLabel distribution after artifact drop:")
print(
    wot_clean.loc[~artifact_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)

print("\nLabel distribution among dropped artifact rows:")
print(
    wot_clean.loc[artifact_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)

# create artifact-cleaned dataset
wot_artifact_clean = (
    wot_clean
    .loc[~artifact_mask]
    .drop(columns=["artifact_status"])
    .reset_index(drop=True)
)

print(f"\nRows before artifact drop: {len(wot_deduped)}")
print(f"Dropped {artifact_mask.sum()} ({artifact_mask.sum() / len(wot_deduped):.1%})")
print(f"Rows after artifact drop: {len(wot_artifact_clean)}")

Artifact status counts:
artifact_status
keep             49579
drop_error         171
drop_html_tag       16
drop_url             7
Name: count, dtype: int64

Artifact rows to drop:
                                                                                                              message  \
11                                                                                                           #ERROR!   
64                                                                                                           #ERROR!   
65                                                     Awww... show me on the tank where the bad arty touched you...   
168    https://www.twitch.tv/malpa_gaming_ha_ha KONKURS NA ZLOTO I PREMIUM TANKS ..GIVEWAY FOR GOLD AND PREMIUM TANK   
917                                                                                                          #ERROR!   
978                                                                                              

In [21]:
# save again and evaluate 
wot_artifact_clean.to_parquet(PROJECT_ROOT / "data/processed_data/wot/wot.parquet", index=False)
preprocessing_impact = eval_step("after_artifacts_removal", preprocessing_impact, datasets = ("WoT",))

                   step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
after_artifacts_removal     WoT 49579      0.53       0.8772        0.6118           0.5181          0.0009       -0.0115           0.0075


In [22]:
# check 
preprocessing_impact

,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4855,0.8610,0.5599,0.4696,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4976,0.8692,0.5921,0.4799,0.0121,0.0322,0.0103
2,after_noneng_drop,WoT,49773,0.4922,0.8683,0.5921,0.4788,-0.0054,0.0000,-0.0011
3,after_duplicates_removal,WoT,42475,0.4919,0.8658,0.5711,0.5091,-0.0003,-0.0210,0.0303
4,after_duplicates_majority_map,WoT,49773,0.5291,0.8778,0.6233,0.5106,0.0372,0.0522,0.0015
5,after_artifacts_removal,WoT,49579,0.5300,0.8772,0.6118,0.5181,0.0009,-0.0115,0.0075


#### Mislabeled data 

In [23]:
# few top rows of the data
wot_artifact_clean.head(10)

,index,message,label
0,30702,no rush,0.0
1,18607,whatever ... watch the replay,0.0
2,32901,useless,1.0
3,25964,3 gunmark,0.0
4,28643,lol,0.0
5,4313,i softened him lol,0.0
6,8765,come,0.0
7,52567,stupid kids grow up,1.0
8,24437,fking light diference,0.0
9,41928,"hori, info ?",0.0


In [24]:
# class distribution check
print("Label distribution after cleaning:")
print(wot_artifact_clean["label"].value_counts(normalize=True).sort_index())

Label distribution after cleaning:
label
0.0    0.811916
1.0    0.137034
2.0    0.043890
3.0    0.005244
4.0    0.001372
5.0    0.000545
Name: proportion, dtype: float64


We might have mislabeled toxicity data that was classified as non-toxic or opposite. We need to detect and fix that as well. 

In [25]:
from cleanlab.filter import find_label_issues
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_predict, StratifiedKFold
import numpy as np

# exclude messages under 3 chars — no signal, cleanlab guesses randomly on them
cl_mask = wot_artifact_clean["message"].str.len() >= 3
cl_df = wot_artifact_clean[cl_mask].reset_index(drop=True)

X = cl_df["message"].values
y = cl_df["label"].astype(int).values

# stratified cv with shuffle — avoids ordering bias in folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7524)

# lightweight pipeline to generate calibrated out-of-fold probabilities
# class_weight="balanced" — needed with imbalanced data so minority class errors are detected
# LR required (not SVC) — cleanlab needs predict_proba, not decision_function
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_df=0.95,
                              sublinear_tf=True, norm="l2",
                              analyzer="word", tokenizer=tokenize, token_pattern=None)),
    ("clf", LogisticRegression(max_iter=1000, random_state=7524, class_weight="balanced")),
])

# each sample predicted by model that never saw it during training
oof_probs = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1)

# normalized_margin more robust than self_confidence on imbalanced data —
# only flags rows where model is confident another class is correct, not just uncertain
issue_idx = find_label_issues(
    labels=y,
    pred_probs=oof_probs,
    return_indices_ranked_by="normalized_margin",
)

predicted = oof_probs[issue_idx].argmax(axis=1)
given = y[issue_idx]

# only trust 0 - {1,2,3} boundary crossings —
# class 4/5 too rare (< 1%) for reliable OOF predictions, causes kill/die false positives
boundary_mask = (
    ((given == 0) & np.isin(predicted, [1, 2, 3]))
    | (np.isin(given, [1, 2, 3]) & (predicted == 0))
)
issue_idx = issue_idx[boundary_mask]

print(f"Suspected mislabeled: {len(issue_idx)} ({len(issue_idx)/len(y):.1%})")

# inspect top 50 — compare original label vs what model predicted
print(cl_df.iloc[issue_idx[:50]][["message", "label"]].assign(
    predicted=oof_probs[issue_idx[:50]].argmax(axis=1)
))

# map issue positions in cl_df back to wot_artifact_clean original index
original_idx = wot_artifact_clean[cl_mask].index[issue_idx]

Suspected mislabeled: 2781 (6.4%)
                           message  label  predicted
41327        they call me an idiot    0.0          1
37108         nicalo idiot and all    0.0          1
8809                 cromvel idiot    0.0          1
38361               idiot campersw    0.0          1
4075                     its a bot    0.0          1
29660                      top bot    0.0          1
6796                     its a bot    0.0          1
23674      wtf i win this and loss    0.0          2
8367                      just bot    0.0          1
7363              top tier top bot    0.0          1
25413               and an afk bot    0.0          1
32693                  t1heavy bot    0.0          1
22714             top tier top bot    0.0          1
14687              S5 was a bot :/    0.0          1
11208               you idiot meds    0.0          1
36990               haftoo be bots    0.0          1
40521                     omg bots    0.0          1
5325        

In [26]:
# copy to preserve original — relabeling wot_clean1, wot_clean stays untouched
wot_clean1 = wot_artifact_clean.copy()
wot_clean1.loc[original_idx, "label"] = oof_probs[issue_idx].argmax(axis=1)

print(f"Relabeled {len(issue_idx)} rows ({len(issue_idx)/len(wot_clean1):.1%})")

Relabeled 2781 rows (5.6%)


In [27]:
# save again and evaluate 
wot_clean1.to_parquet(PROJECT_ROOT / "data/processed_data/wot/wot.parquet", index=False)
preprocessing_impact = eval_step("after_labels_fix", preprocessing_impact, datasets = ("WoT",))

            step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
after_labels_fix     WoT 49579    0.6109       0.9116        0.6969            0.615          0.0809        0.0851           0.0969


In [28]:
preprocessing_impact

,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4855,0.8610,0.5599,0.4696,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4976,0.8692,0.5921,0.4799,0.0121,0.0322,0.0103
2,after_noneng_drop,WoT,49773,0.4922,0.8683,0.5921,0.4788,-0.0054,0.0000,-0.0011
3,after_duplicates_removal,WoT,42475,0.4919,0.8658,0.5711,0.5091,-0.0003,-0.0210,0.0303
4,after_duplicates_majority_map,WoT,49773,0.5291,0.8778,0.6233,0.5106,0.0372,0.0522,0.0015
5,after_artifacts_removal,WoT,49579,0.5300,0.8772,0.6118,0.5181,0.0009,-0.0115,0.0075
6,after_labels_fix,WoT,49579,0.6109,0.9116,0.6969,0.6150,0.0809,0.0851,0.0969


That might be a model inflation. Let's check if gains are real with other model.

In [29]:
from sklearn.svm import LinearSVC

# impact 
preprocessing_impact = eval_step("after_labels_fix_svc", preprocessing_impact,
                                  datasets=("WoT",),
                                  clf=LinearSVC(max_iter=2000, random_state=7524, 
                                                class_weight = "balanced"))
preprocessing_impact

                step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
after_labels_fix_svc     WoT 49579    0.6303       0.9218        0.6359           0.6642          0.0194        -0.061           0.0492


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4855,0.8610,0.5599,0.4696,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4976,0.8692,0.5921,0.4799,0.0121,0.0322,0.0103
2,after_noneng_drop,WoT,49773,0.4922,0.8683,0.5921,0.4788,-0.0054,0.0000,-0.0011
3,after_duplicates_removal,WoT,42475,0.4919,0.8658,0.5711,0.5091,-0.0003,-0.0210,0.0303
4,after_duplicates_majority_map,WoT,49773,0.5291,0.8778,0.6233,0.5106,0.0372,0.0522,0.0015
5,after_artifacts_removal,WoT,49579,0.5300,0.8772,0.6118,0.5181,0.0009,-0.0115,0.0075
6,after_labels_fix,WoT,49579,0.6109,0.9116,0.6969,0.6150,0.0809,0.0851,0.0969
7,after_labels_fix_svc,WoT,49579,0.6303,0.9218,0.6359,0.6642,0.0194,-0.0610,0.0492


There's still a risk that the score gains are inflated, however the data cleaning is real and coherent. 

#### Final verification

In [30]:
# basic data checks
print("=== Shape ===")
print(wot_clean1.shape)

print("\n=== Nulls ===")
print(wot_clean1.isnull().sum())

print("\n=== Overall label distribution ===")
counts = wot_clean1["label"].value_counts().sort_index()
pct = wot_clean1["label"].value_counts(normalize=True).sort_index().mul(100).round(2)
print(pd.DataFrame({"count": counts, "pct%": pct}))

=== Shape ===
(49579, 3)

=== Nulls ===
index      0
message    0
label      0
dtype: int64

=== Overall label distribution ===
       count   pct%
label              
0.0    38993  78.65
1.0     6927  13.97
2.0     3191   6.44
3.0      373   0.75
4.0       68   0.14
5.0       27   0.05


In [31]:
# save to parquet for easier loading in future steps
wot_clean1.to_parquet(PROJECT_ROOT / "data/processed_data/wot/wot.parquet", index=False)

## Dota

**Dota**
| Class | Label |
|---|---|
| 0 | Other (non-toxic) |
| 1 | Ego |
| 2 | Aggression |
| 3 | Impolite |

#### Inspect each split

In [32]:
# import data
train = pd.read_csv(DATA_DIR_DOTA / "CONDA_train.csv")
val = pd.read_csv(DATA_DIR_DOTA / "CONDA_valid.csv")

# inspect each split
for name, df in [("train", train), ("val", val)]:
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}\n")
    print(f"Columns: {list(df.columns)}\n")
    print(f"First few rows of the data:\n{df.head(3)}\n")

--- train ---
Shape: (26921, 10)

Columns: ['Id', 'matchId', 'conversationId', 'utterance', 'chatTime', 'playerSlot', 'playerId', 'intentClass', 'slotClasses', 'slotTokens']

First few rows of the data:
      Id  matchId  conversationId utterance  chatTime  playerSlot  \
0  11263      697            3193      wow!        76           0   
1  13741      843            3809       WTF      1563           5   
2  22125     1412            6199   wpe wpe      2853           1   

                  playerId intentClass slotClasses          slotTokens  
0  ANTS IN MY EYES JOHNSON           O          O            wow (O),   
1                      M.k           O          T            WTF (T),   
2              Acqua Ragia           O        O O   wpe (O), wpe (O),   

--- val ---
Shape: (8974, 10)

Columns: ['Id', 'matchId', 'conversationId', 'utterance', 'chatTime', 'playerSlot', 'playerId', 'intentClass', 'slotClasses', 'slotTokens']

First few rows of the data:
      Id  matchId  conversa

#### Clean, merge and standardise

In [33]:
# map intentClass to numeric label consistent with WoT schema
label_map = {'O': 0, 'E': 1, 'A': 2, 'I': 3}

# add split column and map labels
train["split"] = "train"
val["split"]   = "val"

# merge train + val (no test split in CONDA)
dota = pd.concat([train, val], ignore_index=True)

# keep only relevant columns and rename to match WoT schema
dota = dota[["Id", "utterance", "intentClass", "split"]].rename(columns={
    "Id": "index",
    "utterance":   "message",
    "intentClass": "label",
})

# map string labels to numeric
dota["label"] = dota["label"].map(label_map)

# drop nulls in message (7 train + 1 val)
dota = dota.dropna(subset=["message"]).reset_index(drop=True)

# shape 
print(f"Merged shape: {dota.shape}\n")
print(f"First few rows of the data:\n{dota.head(3)}\n")
print("\nInformation about the dataset:\n")
dota.info()

Merged shape: (35887, 4)

First few rows of the data:
   index  message  label  split
0  11263     wow!      0  train
1  13741      WTF      0  train
2  22125  wpe wpe      0  train


Information about the dataset:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35887 entries, 0 to 35886
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   index    35887 non-null  int64 
 1   message  35887 non-null  object
 2   label    35887 non-null  int64 
 3   split    35887 non-null  object
dtypes: int64(2), object(2)
memory usage: 1.1+ MB


In [34]:
# save raw dota as baseline for ablation
dota.to_parquet(PROJECT_ROOT / "data/processed_data/dota/dota.parquet", index=False)
preprocessing_impact = eval_step("dota_baseline_raw", preprocessing_impact, datasets=("Dota",))

             step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro
dota_baseline_raw    Dota 35887    0.7538       0.8692        0.7915           0.7288


In [35]:
preprocessing_impact

,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4855,0.8610,0.5599,0.4696,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4976,0.8692,0.5921,0.4799,0.0121,0.0322,0.0103
2,after_noneng_drop,WoT,49773,0.4922,0.8683,0.5921,0.4788,-0.0054,0.0000,-0.0011
3,after_duplicates_removal,WoT,42475,0.4919,0.8658,0.5711,0.5091,-0.0003,-0.0210,0.0303
4,after_duplicates_majority_map,WoT,49773,0.5291,0.8778,0.6233,0.5106,0.0372,0.0522,0.0015
5,after_artifacts_removal,WoT,49579,0.5300,0.8772,0.6118,0.5181,0.0009,-0.0115,0.0075
6,after_labels_fix,WoT,49579,0.6109,0.9116,0.6969,0.6150,0.0809,0.0851,0.0969
7,after_labels_fix_svc,WoT,49579,0.6303,0.9218,0.6359,0.6642,0.0194,-0.0610,0.0492
8,dota_baseline_raw,Dota,35887,0.7538,0.8692,0.7915,0.7288,NaN,NaN,NaN


Only `utterance`, `intentClass`, `Id`, and `split` are kept = the remaining columns (matchId, conversationId, chatTime, playerSlot, playerId, slotClasses, slotTokens) carry no linguistic content relevant to toxicity detection. Labels are mapped to integers matching the WoT schema where 0 = non-toxic. Null utterances are dropped - they carry no text signal.

#### Non-english language 

Since we are going to model classifier on english data, it is important for us to detect non-english cases as many as we can. 

In [36]:
# covers all major non-Latin scripts
NON_LATIN_SCRIPT = re.compile(
    r"[\u0400-\u04FF"   # Cyrillic
    r"\u4E00-\u9FFF"    # CJK unified ideographs
    r"\u3400-\u4DBF"    # CJK extension A
    r"\uF900-\uFAFF"    # CJK compatibility ideographs
    r"\u0600-\u06FF"    # Arabic
    r"\u0590-\u05FF"    # Hebrew
    r"\u3040-\u30FF"    # Japanese (Hiragana + Katakana)
    r"\uAC00-\uD7AF"    # Korean (Hangul syllables)
    r"\u1100-\u11FF"    # Korean (Hangul Jamo)
    r"\u0E00-\u0E7F"    # Thai
    r"\u0900-\u097F"    # Devanagari (Hindi)
    r"\u0980-\u09FF"    # Bengali
    r"\u0370-\u03FF"    # Greek
    r"\u10A0-\u10FF"    # Georgian
    r"\u0530-\u058F"    # Armenian
    r"\u1000-\u109F"    # Myanmar
    r"\u1780-\u17FF]"   # Khmer
)

# drop any message containing even 1 non-Latin script char
print(f"Non-english documents:\n{dota[dota['message'].str.contains(NON_LATIN_SCRIPT, regex=True, na=False)].reset_index(drop=True)}\n")

before = len(dota)
dota_en = dota[~dota["message"].str.contains(NON_LATIN_SCRIPT, regex=True, na=False)].reset_index(drop=True)
print(f"Dropped {before - len(dota_en)} ({(before - len(dota_en))/before:.1%})")
print("Label distribution before:")
print(dota["label"].value_counts(normalize=True).sort_index(), "\n")

print("\nLabel distribution after:")
print(dota_en["label"].value_counts(normalize=True).sort_index(), "\n")
print(f"Rows after the drop: {dota_en.shape[0]}")
# save new step again
dota_en.to_parquet(PROJECT_ROOT / "data/processed_data/dota/dota.parquet", index=False)

Non-english documents:
    index                                   message  label  split
0    8390                       s [SEPA] v [SEPA] ツ      0  train
1   23981      sure go mid with bh [SEPA] ¯\_(ツ)_/¯      0  train
2   24030                                       Νοο      0  train
3   36141                           noob [SEPA] กาก      1  train
4   19064                                     ใจยเน      0  train
5   18820                                ညသ တေမိ  ၊      0  train
6   23983                                 ¯\_(ツ)_/¯      0  train
7   27051                              เเ [SEPA] gg      0  train
8    8391                                         ツ      0  train
9   27048                                         ื      0  train
10  37396                                        ำผ      0  train
11  25892                              ไย [SEPA] wp      0  train
12  41950                                       กาก      0    val
13  19066                                     คนไทย  

In [37]:
# save and evaluate non-Latin drop
dota_en.to_parquet(PROJECT_ROOT / "data/processed_data/dota/dota.parquet", index=False)
preprocessing_impact = eval_step("dota_after_nonlatin_drop", preprocessing_impact, datasets=("Dota",))
preprocessing_impact

                    step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
dota_after_nonlatin_drop    Dota 35869    0.7544       0.8691         0.791             0.73          0.0006       -0.0005           0.0012


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4855,0.8610,0.5599,0.4696,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4976,0.8692,0.5921,0.4799,0.0121,0.0322,0.0103
2,after_noneng_drop,WoT,49773,0.4922,0.8683,0.5921,0.4788,-0.0054,0.0000,-0.0011
3,after_duplicates_removal,WoT,42475,0.4919,0.8658,0.5711,0.5091,-0.0003,-0.0210,0.0303
4,after_duplicates_majority_map,WoT,49773,0.5291,0.8778,0.6233,0.5106,0.0372,0.0522,0.0015
5,after_artifacts_removal,WoT,49579,0.5300,0.8772,0.6118,0.5181,0.0009,-0.0115,0.0075
6,after_labels_fix,WoT,49579,0.6109,0.9116,0.6969,0.6150,0.0809,0.0851,0.0969
7,after_labels_fix_svc,WoT,49579,0.6303,0.9218,0.6359,0.6642,0.0194,-0.0610,0.0492
8,dota_baseline_raw,Dota,35887,0.7538,0.8692,0.7915,0.7288,NaN,NaN,NaN
9,dota_after_nonlatin_drop,Dota,35869,0.7544,0.8691,0.7910,0.7300,0.0006,-0.0005,0.0012


In [38]:
import re
import pandas as pd
from lingua import Language, LanguageDetectorBuilder

# detecting Latin-script but likely non-English messages

# use all Latin-script languages supported by Lingua
# English must stay inside the detector, otherwise English text gets forced into another language
LATIN_LANGUAGES = list(Language.all_with_latin_script())

# detector for Latin-script language detection
# minimum_relative_distance: if top languages are too close, Lingua returns None instead of guessing
detector = (
    LanguageDetectorBuilder
    .from_languages(*LATIN_LANGUAGES)
    .with_minimum_relative_distance(0.25)
    .build()
)

# safety tokens: if these appear, keep the message because it may be gaming/chat English or mixed slang
SAFETY_TOKENS = {
    # gaming / chat
    "gg", "ez", "wp", "gl", "hf", "glhf", "afk", "brb", "lol", "lmao", "lmfao",
    "omg", "wtf", "stfu", "noob", "n00b", "bot", "team", "report", "ban",
    "kick", "mute", "trash", "toxic", "troll", "feed", "feeder", "int", "throw",
    "rank", "elo", "kill", "kys", "die", "fps", "lag", "ping", "server",
    "wot", "dota", "fortnite", "fn",

    # English / toxicity cues
    "you", "your", "you're", "u", "ur", "are", "is", "this", "that", "the",
    "and", "not", "dont", "don't", "fuck", "fucking", "fuk", "fck", "shit",
    "bitch", "ass", "asshole", "idiot", "stupid", "dumb", "moron", "loser",
    "suck", "sucks"
}

TOKEN_RE = re.compile(r"[a-z0-9']+", flags=re.IGNORECASE)


def get_tokens(text):
    return set(TOKEN_RE.findall(str(text).lower()))


def has_safety_token(text):
    return bool(get_tokens(text) & SAFETY_TOKENS)


# function to classify what to do with each Latin-script message
def latin_language_filter_status(text):
    text_str = str(text).strip()
    tokens = get_tokens(text_str)

    # too short - keep
    # language detection is unreliable on short gaming/chat messages
    if len(text_str) < 50 or len(tokens) < 6:
        return "keep_short"

    # possible gaming/chat/English false positive - keep
    if has_safety_token(text_str):
        return "keep_possible_gaming_or_mixed"

    lang = detector.detect_language_of(text_str)

    # ambiguous - keep
    if lang is None:
        return "keep_ambiguous"

    # English - keep
    if lang == Language.ENGLISH:
        return "keep_english"

    # confident Latin-script non-English - drop
    return f"drop_latin_nonenglish_{lang.name.lower()}"


# apply detection
dota_en_lang = dota_en.copy()
dota_en_lang["lang_filter_status"] = dota_en_lang["message"].apply(latin_language_filter_status)

# inspect groups
print("Language filter status counts:")
print(dota_en_lang["lang_filter_status"].value_counts().head(30))


# check ambiguous cases
# these are messages where Lingua did not confidently choose a language
ambiguous_mask = dota_en_lang["lang_filter_status"].eq("keep_ambiguous")

print(f"\nAmbiguous Latin-script language documents kept: {ambiguous_mask.sum()}")

with pd.option_context("display.max_rows", 300, "display.max_colwidth", 180):
    print(
        "\nAmbiguous documents kept for now:\n",
        dota_en_lang.loc[
            ambiguous_mask,
            ["message", "label", "lang_filter_status"]
        ]
    )

print("\nLabel distribution among ambiguous rows:")
print(
    dota_en_lang.loc[ambiguous_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)


# check possible gaming / mixed-language false positives
# these are messages that Lingua might classify as non-English, but contain gaming/chat/English cues
mixed_mask = dota_en_lang["lang_filter_status"].eq("keep_possible_gaming_or_mixed")

print(f"\nPossible gaming or mixed-language documents kept: {mixed_mask.sum()}")

with pd.option_context("display.max_rows", 300, "display.max_colwidth", 180):
    print(
        "\nPossible gaming/mixed documents kept for now:\n",
        dota_en_lang.loc[
            mixed_mask,
            ["message", "label", "lang_filter_status"]
        ]
    )

print("\nLabel distribution among possible gaming/mixed rows:")
print(
    dota_en_lang.loc[mixed_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)


# drop only confident Latin-script non-English rows
drop_mask = dota_en_lang["lang_filter_status"].str.startswith("drop_latin_nonenglish")

with pd.option_context("display.max_rows", 300, "display.max_colwidth", 180):
    print(
        "\nConfident Latin-script non-English documents to drop:\n",
        dota_en_lang.loc[
            drop_mask,
            ["message", "label", "lang_filter_status"]
        ]
    )

print(f"\nHow many confident Latin-script non-English documents: {drop_mask.sum()}")


# label distribution sanity check
dropped_latin_nonenglish = dota_en_lang.loc[drop_mask]

print("\nLabel distribution before Latin non-English drop:")
print(dota_en["label"].value_counts(normalize=True).sort_index())

print("\nLabel distribution after Latin non-English drop:")
print(
    dota_en_lang.loc[~drop_mask, "label"]
    .value_counts(normalize=True)
    .sort_index()
)

print("\nLabel distribution among dropped rows:")
print(
    dropped_latin_nonenglish["label"]
    .value_counts(normalize=True)
    .sort_index()
)


# create dataset after Latin non-English drop
# keep ambiguous and mixed/gaming rows
dota_lang_clean = (
    dota_en_lang
    .loc[~drop_mask]
    .drop(columns=["lang_filter_status"])
    .reset_index(drop=True)
)

print(f"\nRows before Latin non-English drop: {len(dota_en)}")
print(f"Dropped {drop_mask.sum()} ({drop_mask.sum() / len(dota_en):.1%})")
print(f"Rows after Latin non-English drop: {len(dota_lang_clean)}")

Language filter status counts:
lang_filter_status
keep_short                          33514
keep_possible_gaming_or_mixed        2019
keep_ambiguous                        191
keep_english                          129
drop_latin_nonenglish_portuguese        6
drop_latin_nonenglish_tagalog           6
drop_latin_nonenglish_spanish           3
drop_latin_nonenglish_sotho             1
Name: count, dtype: int64

Ambiguous Latin-script language documents kept: 191

Ambiguous documents kept for now:
                                                                                                                                                                                    message  \
212                                                                                                                             hahaha [SEPA] im only 1 there no one came :( [SEPA] hahaha   
451                                                                                                                    

In [39]:
# save to parquet
dota_lang_clean.to_parquet(PROJECT_ROOT / "data/processed_data/dota/dota.parquet", index=False)

# evaluate impact 
preprocessing_impact = eval_step("after_noneng_drop", preprocessing_impact, datasets = ("Dota",))
preprocessing_impact

             step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
after_noneng_drop    Dota 35853    0.7514       0.8682        0.7885           0.7267          -0.003       -0.0025          -0.0033


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4855,0.8610,0.5599,0.4696,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4976,0.8692,0.5921,0.4799,0.0121,0.0322,0.0103
2,after_noneng_drop,WoT,49773,0.4922,0.8683,0.5921,0.4788,-0.0054,0.0000,-0.0011
3,after_duplicates_removal,WoT,42475,0.4919,0.8658,0.5711,0.5091,-0.0003,-0.0210,0.0303
4,after_duplicates_majority_map,WoT,49773,0.5291,0.8778,0.6233,0.5106,0.0372,0.0522,0.0015
5,after_artifacts_removal,WoT,49579,0.5300,0.8772,0.6118,0.5181,0.0009,-0.0115,0.0075
6,after_labels_fix,WoT,49579,0.6109,0.9116,0.6969,0.6150,0.0809,0.0851,0.0969
7,after_labels_fix_svc,WoT,49579,0.6303,0.9218,0.6359,0.6642,0.0194,-0.0610,0.0492
8,dota_baseline_raw,Dota,35887,0.7538,0.8692,0.7915,0.7288,NaN,NaN,NaN
9,dota_after_nonlatin_drop,Dota,35869,0.7544,0.8691,0.7910,0.7300,0.0006,-0.0005,0.0012


Decrease in CV metrics is within a noise range.

#### Data Artifacts


In [40]:
import re
import html as html_lib

# artifact cleaning for Dota — same pattern as WoT
# input: dota_en (after SEPA strip and language filter)
dota_clean = dota_lang_clean.copy()
dota_clean["message"] = dota_clean["message"].fillna("").astype(str)

# strip [SEPA] conversation separator — data collection artifact, no linguistic value
dota_clean["message"] = dota_clean["message"].str.replace(r"\s*\[SEPA\]\s*", " ", regex=True).str.strip()
print(f"[SEPA] remaining: {dota_clean['message'].str.contains(r'[SEPA]', regex=False).sum()}")

# url links 
URL_RE = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
# html tags
HTML_ENTITY_RE = re.compile(r"&[a-zA-Z]+;|&#\d+;|&#x[0-9a-fA-F]+;")
# keyboard mash: 20+ consecutive non-whitespace chars with no vowels or mostly symbols
KEYBOARD_MASH_RE = re.compile(r"\S{20,}")

# function for classifying artifact status of each message
def artifact_status(text):
    text = str(text)
    if URL_RE.search(text):
        return "drop_url"
    if HTML_ENTITY_RE.search(text):
        return "drop_html_entity"
    if KEYBOARD_MASH_RE.search(text):
        return "drop_keyboard_mash"
    return "keep"

# apply artifact detection
dota_clean["artifact_status"] = dota_clean["message"].apply(artifact_status)

print("Artifact status counts:")
print(dota_clean["artifact_status"].value_counts())

# mask for rows to drop
artifact_mask = dota_clean["artifact_status"].ne("keep")

print("\nArtifact rows to drop:")
with pd.option_context("display.max_rows", 50, "display.max_colwidth", 120):
    print(dota_clean.loc[artifact_mask, ["message", "label", "artifact_status"]])

print("\nLabel distribution before artifact drop:")
print(dota_lang_clean["label"].value_counts(normalize=True).sort_index())

print("\nLabel distribution after artifact drop:")
print(dota_clean.loc[~artifact_mask, "label"].value_counts(normalize=True).sort_index())

# create final cleaned dataset after dropping artifacts
dota_artifact_clean = (
    dota_clean
    .loc[~artifact_mask]
    .drop(columns=["artifact_status"])
    .reset_index(drop=True)
)

print(f"\nRows before: {len(dota_lang_clean)}")
print(f"Dropped: {artifact_mask.sum()} ({artifact_mask.sum()/len(dota_lang_clean):.1%})")
print(f"Rows after: {len(dota_artifact_clean)}")

[SEPA] remaining: 0
Artifact status counts:
artifact_status
keep                  35749
drop_keyboard_mash       97
drop_url                  7
Name: count, dtype: int64

Artifact rows to drop:
                                                       message  label  \
733    lol these 3 eggs these eggs are tsuyuyuy,tsokotomotoko.      0   
743                                       lollllllllllllllllll      0   
821                                       SOOOOOOOOOOOOOOOOBAD      0   
1072                              aaahajahahahahhahahahahahaha      0   
1124                                  ooooooooooooooooooooooom      0   
...                                                        ...    ...   
33398                   peeeeeee sorry peeeeeeeeeeeeeeeeeeeeee      0   
33518                    wanted me that badddddddddddddddddddd      0   
34697                        fattttttttttttttttttttttttttttttt      0   
35405                     BLYAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAT      0   
357

In [41]:
# save to parquet
dota_artifact_clean.to_parquet(PROJECT_ROOT / "data/processed_data/dota/dota.parquet", index=False)

# evaluate impact 
preprocessing_impact = eval_step("after_artifact_drop", preprocessing_impact, datasets = ("Dota",))
preprocessing_impact

               step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
after_artifact_drop    Dota 35749    0.7574       0.8717         0.792           0.7334           0.006        0.0035           0.0067


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4855,0.8610,0.5599,0.4696,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4976,0.8692,0.5921,0.4799,0.0121,0.0322,0.0103
2,after_noneng_drop,WoT,49773,0.4922,0.8683,0.5921,0.4788,-0.0054,0.0000,-0.0011
3,after_duplicates_removal,WoT,42475,0.4919,0.8658,0.5711,0.5091,-0.0003,-0.0210,0.0303
4,after_duplicates_majority_map,WoT,49773,0.5291,0.8778,0.6233,0.5106,0.0372,0.0522,0.0015
5,after_artifacts_removal,WoT,49579,0.5300,0.8772,0.6118,0.5181,0.0009,-0.0115,0.0075
6,after_labels_fix,WoT,49579,0.6109,0.9116,0.6969,0.6150,0.0809,0.0851,0.0969
7,after_labels_fix_svc,WoT,49579,0.6303,0.9218,0.6359,0.6642,0.0194,-0.0610,0.0492
8,dota_baseline_raw,Dota,35887,0.7538,0.8692,0.7915,0.7288,NaN,NaN,NaN
9,dota_after_nonlatin_drop,Dota,35869,0.7544,0.8691,0.7910,0.7300,0.0006,-0.0005,0.0012


#### Duplicates


In [42]:
# look at dataset with duplicates
dota_dup = dota_artifact_clean[dota_artifact_clean.duplicated(subset="message", keep=False)].sort_values("message")
dota_dup.head(10)

,index,message,label,split
17567,11136,!,0,train
26805,38158,!,0,train
28681,7592,!,0,val
19074,42692,!,0,train
23685,24950,!,0,train
10894,32061,!,0,train
9417,35992,!,0,train
33203,1008,!!,0,val
17582,6806,!!,0,train
14375,31944,!!!!,0,train


In [43]:
# top 30 most duplicated messages
dota_artifact_clean["message"].value_counts(ascending=False).head(30)

message
gg        1466
lol        691
?          445
ggwp       355
haha       348
gg wp      316
GG         291
ez         274
LOL        192
ty         160
:D         147
hahaha     135
wp         133
XD          96
GGWP        91
ok          82
EZ          81
+           78
Gg          74
:)          73
wtf         70
rofl        65
xD          60
ez mid      59
wait        59
end         58
HAHA        58
g           55
:(          55
)           54
Name: count, dtype: int64

In [44]:
# find conflicting labels
conflicts_d = dota_artifact_clean.groupby("message")["label"].nunique()
conflicts_d = conflicts_d[conflicts_d > 1]
conflict_rows_d = dota_artifact_clean[dota_artifact_clean["message"].isin(conflicts_d.index)].sort_values("message")
conflict_pct_d = len(conflict_rows_d) / len(dota_artifact_clean) * 100
print(f"Messages with conflicting labels: {len(conflicts_d)}")
print(f"Proportion: {conflict_pct_d:.2f}%")
print(f"Rows affected: {len(conflict_rows_d)}")

Messages with conflicting labels: 147
Proportion: 13.71%
Rows affected: 4900


In [45]:
# check conflicting rows
conflict_rows_d

,index,message,label,split
9238,21192,),0,train
4506,38062,),0,train
33727,37984,),0,val
18484,34207,),0,train
30978,41690,),0,val
...,...,...,...,...
17468,7192,zz,0,train
957,22280,zz,0,train
853,30365,zz,0,train
31593,26815,zz,0,val


Same as WoT - annotation noise. Dropping would lose training signal. Assign majority label instead.


In [46]:
# majority label for all messages
majority_label_d = dota_artifact_clean.groupby("message")["label"].agg(lambda x: x.value_counts().index[0])

# remap all labels to majority vote
dota_deduped = dota_artifact_clean.copy()
dota_deduped["label"] = dota_deduped["message"].map(majority_label_d)

# check former conflicting messages
dota_deduped[dota_deduped["message"].isin(conflicts_d.index)][["message", "label"]].head(55)

,message,label
1,WTF,0
3,hahaha,0
4,wtf,0
22,ggwp,0
35,gg,0
39,?,0
48,gg,0
52,ez,3
58,ez game,3
71,HAHA,0


In [47]:
# save and evaluate majority map
dota_deduped.to_parquet(PROJECT_ROOT / "data/processed_data/dota/dota.parquet", index=False)
preprocessing_impact = eval_step("dota_after_majority_map", preprocessing_impact, datasets=("Dota",))
preprocessing_impact

                   step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
dota_after_majority_map    Dota 35749    0.7666       0.8774        0.7961           0.7451          0.0092        0.0041           0.0117


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4855,0.8610,0.5599,0.4696,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4976,0.8692,0.5921,0.4799,0.0121,0.0322,0.0103
2,after_noneng_drop,WoT,49773,0.4922,0.8683,0.5921,0.4788,-0.0054,0.0000,-0.0011
3,after_duplicates_removal,WoT,42475,0.4919,0.8658,0.5711,0.5091,-0.0003,-0.0210,0.0303
4,after_duplicates_majority_map,WoT,49773,0.5291,0.8778,0.6233,0.5106,0.0372,0.0522,0.0015
5,after_artifacts_removal,WoT,49579,0.5300,0.8772,0.6118,0.5181,0.0009,-0.0115,0.0075
6,after_labels_fix,WoT,49579,0.6109,0.9116,0.6969,0.6150,0.0809,0.0851,0.0969
7,after_labels_fix_svc,WoT,49579,0.6303,0.9218,0.6359,0.6642,0.0194,-0.0610,0.0492
8,dota_baseline_raw,Dota,35887,0.7538,0.8692,0.7915,0.7288,NaN,NaN,NaN
9,dota_after_nonlatin_drop,Dota,35869,0.7544,0.8691,0.7910,0.7300,0.0006,-0.0005,0.0012


That is the best we can do and it actually improves the score as well. 


#### Mislabeled data 

In [48]:
# few top rows of the data
dota_deduped.head(5)

,index,message,label,split
0,11263,wow!,0,train
1,13741,WTF,0,train
2,22125,wpe wpe,0,train
3,6453,hahaha,0,train
4,9644,wtf,0,train


In [49]:
# class distribution check
print("Label distribution after cleaning:")
print(dota_deduped["label"].value_counts(normalize=True).sort_index())

Label distribution after cleaning:
label
0    0.741503
1    0.130913
2    0.064645
3    0.062939
Name: proportion, dtype: float64


We might have mislabeled toxicity data that was classified as non-toxic or opposite. We need to detect and fix that as well. 

In [50]:
from cleanlab.filter import find_label_issues
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_predict, StratifiedKFold
import numpy as np

# exclude messages under 3 chars — no signal, cleanlab guesses randomly on them
cl_mask = dota_deduped["message"].str.len() >= 3
cl_df = dota_deduped[cl_mask].reset_index(drop=True)
    
X = cl_df["message"].values
y = cl_df["label"].astype(int).values

# stratified cv with shuffle — avoids ordering bias in folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7524)

# lightweight pipeline to generate calibrated out-of-fold probabilities
# class_weight="balanced" — needed with imbalanced data so minority class errors are detected
# LR required (not SVC) — cleanlab needs predict_proba, not decision_function
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_df=0.95,
                              sublinear_tf=True, norm="l2",
                              analyzer="word", tokenizer=tokenize, token_pattern=None)),
    ("clf", LogisticRegression(max_iter=1000, random_state=7524, class_weight="balanced")),
])

# each sample predicted by model that never saw it during training
oof_probs = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1)

# normalized_margin more robust than self_confidence on imbalanced data —
# only flags rows where model is confident another class is correct, not just uncertain
issue_idx = find_label_issues(
    labels=y,
    pred_probs=oof_probs,
    return_indices_ranked_by="normalized_margin",
)

predicted = oof_probs[issue_idx].argmax(axis=1)
given = y[issue_idx]

# only trust 0-toxic boundary crossings — inter-class noise not worth relabeling
boundary_mask = (given == 0) != (predicted == 0)
issue_idx = issue_idx[boundary_mask]
print(f"Suspected mislabeled: {len(issue_idx)} ({len(issue_idx)/len(y):.1%})")

# inspect top 50 — compare original label vs what model predicted
print(cl_df.iloc[issue_idx[:50]][["message", "label"]].assign(
    predicted=oof_probs[issue_idx[:50]].argmax(axis=1)
))

# map issue positions in cl_df back to dota_deduped original index
original_idx = dota_deduped[cl_mask].index[issue_idx]

Suspected mislabeled: 1983 (6.5%)
                                                 message  label  predicted
6590                                                 EZ^      0          3
21911                                              NOOB!      0          1
1922                                             im noob      0          1
19908                                       spectre noob      0          1
6115                                                 EZ)      0          3
4612                                                 ez&      0          3
13177                                            oh shit      0          1
14341                       see that shit is just spooky      0          1
3030                                 we dont give a fuck      0          1
9980               why the fuck you lion you always lion      0          1
20802                                    Intense as fuck      0          1
26230                                      WHAT THE FUCK      0   

Most of them are mislabeled. Let's replace labels instead of dropping. 


In [51]:
# copy to preserve original — relabeling dota_clean1, dota_deduped stays untouched
dota_clean1 = dota_deduped.copy()
dota_clean1.loc[original_idx, "label"] = oof_probs[issue_idx].argmax(axis=1)

print(f"Relabeled {len(issue_idx)} rows ({len(issue_idx)/len(dota_clean1):.1%})")

Relabeled 1983 rows (5.5%)


In [52]:
# save and evaluate labels fix
dota_clean1.to_parquet(PROJECT_ROOT / "data/processed_data/dota/dota.parquet", index=False)
preprocessing_impact = eval_step("dota_after_labels_fix", preprocessing_impact, datasets=("Dota",))

                 step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
dota_after_labels_fix    Dota 35749    0.8488       0.9176        0.8736           0.8306          0.0822        0.0775           0.0855


In [53]:
# check 
preprocessing_impact

,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4855,0.8610,0.5599,0.4696,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4976,0.8692,0.5921,0.4799,0.0121,0.0322,0.0103
2,after_noneng_drop,WoT,49773,0.4922,0.8683,0.5921,0.4788,-0.0054,0.0000,-0.0011
3,after_duplicates_removal,WoT,42475,0.4919,0.8658,0.5711,0.5091,-0.0003,-0.0210,0.0303
4,after_duplicates_majority_map,WoT,49773,0.5291,0.8778,0.6233,0.5106,0.0372,0.0522,0.0015
5,after_artifacts_removal,WoT,49579,0.5300,0.8772,0.6118,0.5181,0.0009,-0.0115,0.0075
6,after_labels_fix,WoT,49579,0.6109,0.9116,0.6969,0.6150,0.0809,0.0851,0.0969
7,after_labels_fix_svc,WoT,49579,0.6303,0.9218,0.6359,0.6642,0.0194,-0.0610,0.0492
8,dota_baseline_raw,Dota,35887,0.7538,0.8692,0.7915,0.7288,NaN,NaN,NaN
9,dota_after_nonlatin_drop,Dota,35869,0.7544,0.8691,0.7910,0.7300,0.0006,-0.0005,0.0012


That might be a model inflation. Let's check if gains are real with other model. 


In [54]:
# SVC circularity check
preprocessing_impact = eval_step("dota_after_labels_fix_svc", preprocessing_impact,
                                  datasets=("Dota",), clf=LinearSVC(max_iter=2000, random_state=7524, class_weight="balanced"))

                     step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
dota_after_labels_fix_svc    Dota 35749    0.8703       0.9313        0.8743           0.8676          0.0215        0.0007            0.037


In [55]:
preprocessing_impact

,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4855,0.8610,0.5599,0.4696,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4976,0.8692,0.5921,0.4799,0.0121,0.0322,0.0103
2,after_noneng_drop,WoT,49773,0.4922,0.8683,0.5921,0.4788,-0.0054,0.0000,-0.0011
3,after_duplicates_removal,WoT,42475,0.4919,0.8658,0.5711,0.5091,-0.0003,-0.0210,0.0303
4,after_duplicates_majority_map,WoT,49773,0.5291,0.8778,0.6233,0.5106,0.0372,0.0522,0.0015
5,after_artifacts_removal,WoT,49579,0.5300,0.8772,0.6118,0.5181,0.0009,-0.0115,0.0075
6,after_labels_fix,WoT,49579,0.6109,0.9116,0.6969,0.6150,0.0809,0.0851,0.0969
7,after_labels_fix_svc,WoT,49579,0.6303,0.9218,0.6359,0.6642,0.0194,-0.0610,0.0492
8,dota_baseline_raw,Dota,35887,0.7538,0.8692,0.7915,0.7288,NaN,NaN,NaN
9,dota_after_nonlatin_drop,Dota,35869,0.7544,0.8691,0.7910,0.7300,0.0006,-0.0005,0.0012


There's still a risk that the score gains are inflated, however the data is now cleaner. 


#### Final verification

In [56]:
print("=== Shape ===")
print(dota_clean1.shape)

print("=== Nulls ===")
print(dota_clean1.isnull().sum())

print("=== Label distribution ===")
counts = dota_clean1["label"].value_counts().sort_index()
pct = dota_clean1["label"].value_counts(normalize=True).sort_index().mul(100).round(1)
label_names = {0: "Other (O)", 1: "Ego (E)", 2: "Aggression (A)", 3: "Impolite (I)"}
print(pd.DataFrame({"count": counts, "pct%": pct}).rename(index=label_names))

=== Shape ===
(35749, 4)
=== Nulls ===
index      0
message    0
label      0
split      0
dtype: int64
=== Label distribution ===
                count  pct%
label                      
Other (O)       25713  71.9
Ego (E)          4790  13.4
Aggression (A)   2711   7.6
Impolite (I)     2535   7.1


In [57]:
# save cleaned dota to parquet for easier loading in future steps
dota_clean1.to_parquet(PROJECT_ROOT / "data/processed_data/dota/dota.parquet", index=False)

In [58]:
# save our preprocessing rounds
preprocessing_impact.to_csv(
    PROJECT_ROOT / "data/results/cleaning_ablation.csv", 
    index=False
)
print("Saved cleaning_ablation.csv")

Saved cleaning_ablation.csv
